In [ ]:
# importing libraries like folium,webbrowser,os,osmnx,networkx to open,visualize map and calculate road distances or geodesic distances between different locations
import folium
import webbrowser
import os
from shapely.geometry import Point
import osmnx as ox
import networkx as nx
import pyproj      # Bundled with osmnx, handles ellipsoidal calculations

# Source: Stack overflow, Github 
def calculate_geodesic_distance(lat1, lon1, lat2, lon2):
    """
    Calculates the high-precision geodesic distance between two points 
    on the WGS84 ellipsoidal surface using pyproj.
    """
    # Load standard WGS84 ellipsoid
    geod = pyproj.Geod(ellps="WGS84")
    
    # pyproj expects coordinates in (Longitude, Latitude) order
    _, _, dist_meters = geod.inv(lon1, lat1, lon2, lat2)
    
    # Convert meters to kilometers
    return dist_meters / 1000.0


# Source:"https://osmnx.readthedocs.io/en/stable/internals-reference.html","https://networkx.org/documentation/stable/reference/algorithms/index.html"
def get_actual_road_distance(lat1, lon1, lat2, lon2,graph,graph_proj,geodesic_dist):
    """
    Calculates the shortest drivable road distance using OSMnx and NetworkX.
    """

    try:
      # Creates Shapely Point objects for the two input coordinates.
      # Note: Points are defined in (longitude, latitude) order.
      point1 = Point(lon1, lat1)
      point2 = Point(lon2, lat2)

      # Projects the input points from the original geographic CRS (degrees) into the same projected CRS as the road network graph (meters).
      point1_proj, _ = ox.projection.project_geometry(point1, crs=graph.graph['crs'], to_crs=graph_proj.graph['crs'])
      point2_proj, _ = ox.projection.project_geometry(point2, crs=graph.graph['crs'], to_crs=graph_proj.graph['crs'])
      
      # Map input coordinates to the nearest network nodes
      start_node = ox.nearest_nodes(graph_proj, X=point1_proj.x, Y=point1_proj.y) 
      end_node = ox.nearest_nodes(graph_proj, X=point2_proj.x, Y=point2_proj.y)


      # Computing shortest path and extracting edge attributes
      shortest_route = nx.shortest_path_length(graph_proj, start_node, end_node, weight="length") 

      # Sum segment lengths (meters) and convert to kilometers 
      actual_road_km = shortest_route/ 1000 
      
      # Road distance cannot be shorter than geodesic distance.
    # If this occurs, treat the routing result as invalid and fallback to geodesic distance.
      if actual_road_km < geodesic_dist:
        print("Invalid route detected. Using geodesic distance.")
        return geodesic_dist, "Geodesic"

      return actual_road_km,"Actual"
    
    except nx.NetworkXNoPath:
        # Road network exists but no valid path connects the locations.
        print("No road path exists between these locations, using geodesic distance.")
        return geodesic_dist,"Geodesic"
    
    # Road-network routing unavailable due to API, network,download, or processing limitations.
    except Exception as e:
        print(f"Road routing unavailable: {e}")
        # Fallback to geodesic distance
        return geodesic_dist,"Geodesic"


def dynamic_multi_map(locations_list,dis,distance_type):
    """
    Generates an interactive Folium map marking the selected locations 
    and saves it as an auto-opening HTML file.
    """

    latitude1 = locations_list[0]['latitude']
    longitude1 = locations_list[0]['longitude']

    # Use standard Google Maps tile layers
    google_tiles_url ="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}" 
    map_point = folium.Map(                    
        location=[latitude1,longitude1],
        zoom_start=4,
        tiles=google_tiles_url,
        attr='Google'
    )
    # Add markers with calculated distance popups
    for loc in locations_list:                 
        folium.Marker(
            location=[loc['latitude'], loc['longitude']],
            tooltip=f"{distance_type} Distance between two locations is:{dis:.2f} km",
            icon=folium.Icon(color='red')
        ).add_to(map_point)
    file_name="actual_distance.html"
    map_point.save(file_name)               # Converts the map into HTML and then saves it as a local file
    webbrowser.open('file://' + os.path.realpath(file_name)) # Finds the absolute computer storage path of HTML file and launches it automatically inside web browser


if __name__ == "__main__":
   
    try:
        print("--- Location 1 ---")
        lat1 = float(input('Enter latitude: '))
        lon1 = float(input('Enter longitude: '))
        print(f'Latitude:{lat1}  Longitude:{lon1}')
        print("\n--- Location 2 ---")
        lat2 = float(input('Enter latitude: '))
        lon2 = float(input('Enter longitude: '))
        print(f'Latitude:{lat2}  Longitude:{lon2}')

        # Validate coordinate ranges
        if not (-90 <= lat1 <= 90 and -90 <= lat2 <= 90 and -180 <= lon1 <= 180 and -180 <= lon2 <= 180):
            raise ValueError("Coordinates out of valid range.")
        
        user_locations = [
            { "latitude": lat1, "longitude": lon1},
            {"latitude": lat2, "longitude": lon2}
        ]

        geodesic_dist = calculate_geodesic_distance(lat1, lon1, lat2, lon2)

        # Buffer boundaries to ensure network coverage around endpoints
        padding=max(0.05, min(0.3, geodesic_dist / 50))
        north = max(lat1, lat2) + padding  
        south = min(lat1, lat2) - padding
        east = max(lon1, lon2) + padding
        west = min(lon1, lon2) - padding

        # Downloads the drivable road network within that bounding box from OpenStreetMap
        graph = ox.graph_from_bbox(bbox = (west, south, east, north), network_type="drive")

        if graph.number_of_edges() == 0:
          raise ValueError("No drivable roads found in this specific coordinate region.")
        
      
        # Projects the road network graph into a suitable planar coordinate system (CRS).
        graph_proj = ox.project_graph(graph)

        dis,distance_type =get_actual_road_distance(lat1, lon1, lat2, lon2,graph,graph_proj,geodesic_dist)
        if distance_type=="Actual":
           print(f"Actual Road Distance between Location 1 and Location 2 is: {dis:.2f} km")
        else:
            print(f"Geodesic Distance between Location 1 and Location 2 is: {dis:.2f} km")
        dynamic_multi_map(user_locations,dis,distance_type)


    except ValueError as e:
        print(f"ValueError: {e}")
        if 'geodesic_dist' in locals():
          print(f"Geodesic distance between Location 1 and Location 2 is: {geodesic_dist:.2f} km")
          dynamic_multi_map(user_locations, geodesic_dist, "Geodesic")


    except Exception as e:
       # If road-network routing cannot be performed due to API/network limitations, return geodesic distance as fallback.
       print(f"Routing/Network Error: {e}")
       if 'geodesic_dist' in locals():
          print(f"Geodesic distance between Location 1 and Location 2 is: {geodesic_dist:.2f} km")
          dynamic_multi_map(user_locations, geodesic_dist, "Geodesic") 

--- Location 1 ---
Latitude:23.8863  Longitude:45.0811

--- Location 2 ---
Latitude:19.1326  Longitude:72.911


c:\Users\SNETA GHOSH\py312\Lib\site-packages\osmnx\_overpass.py:271: UserWarning: This area is 720 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


Routing/Network Error: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by SSLError(SSLError(1, '[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:1000)')))
Geodesic distance between Location 1 and Location 2 is: 2926.17 km
